# **CONDITIONAL CHAINS**

RunnableBranch is an if-else for chains.
It evaluates conditions on the input and routes to a different sub-chain depending on the result.
Here: classify a movie review as positive/negative, then generate platform-appropriate content.

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


In [ ]:
from pydantic import BaseModel
from typing import Literal

# Literal type restricts the field to ONLY these exact string values
# The LLM cannot hallucinate other values — it must choose 'positive' or 'negative'
class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

# Create a structured-output version of the LLM that always returns llm_schema
llm_structured_output = llm_groq.with_structured_output(llm_schema)


# **CHAIN WITH Conditional Chains**

In [ ]:
# TASK 1 — Prompt: ask the LLM to classify a review as positive or negative
# {input} = the raw user review text
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human",  "Please categorize the movie review as positive or negative : {input}")
])


In [ ]:
# TASK 2 — Use the structured LLM so output is always an llm_schema object
# (not a free-form string that might say 'mostly positive' etc.)
llm_structured_output = llm_groq.with_structured_output(llm_schema)


In [ ]:
# TASK 3 — Extract just the flag string from the Pydantic object
from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) -> str:
    # .model_dump() converts the Pydantic object to a plain dict
    # ['movie_summary_flag'] picks out just the classification string
    # Returns: 'positive' or 'negative'
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)


### **Conditional Branch 1 — LinkedIn post (for positive reviews)**

In [ ]:
# This branch runs when the review is POSITIVE
# {text} here receives the raw classification string ('positive') as input
# Note: in real use you'd likely pass the original review too — this is simplified for learning
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human",  "Create a post for the following text for LinkedIn: {text}")
])

llm_groq   = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_groq | str_parser


### **Conditional Branch 2 — Instagram post (for negative reviews)**

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

# This branch runs when the review is NEGATIVE
# RunnableBranch passes the raw string ('negative') as input to this function
def insta_chain(text: dict):
    # In this flow the 'text' arg arrives as a plain string from pydantic_json_lambda
    # RunnableBranch routes it here, so 'text' = 'negative'
    text = text["text"]  # NOTE: if input is a plain string, use: text = text

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        ("human",  "Create a post for the following text for Instagram: {text}")
    ])
    llm_groq_inner   = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    str_parser_inner = StrOutputParser()

    chain_insta = insta_prompt | llm_groq_inner | str_parser_inner
    result = chain_insta.invoke(text)
    return result

insta_chain_runnable = RunnableLambda(insta_chain)


### **Final Orchestration**

In [ ]:
# RunnableBranch is an if-elif-else structure:
#   (condition_function, chain_to_run_if_true),   ← can have many of these
#   default_chain                                  ← runs when NO condition matches
#
# Each condition_function receives the current input and returns True/False
# Here: if 'positive' is in the flag string → LinkedIn; else → Instagram
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),  # condition 1 + its chain
    insta_chain_runnable                           # default branch (negative)
)

# Full pipeline:
#  prompt_template        → fill {input} with the review text
#  llm_structured_output  → classify → llm_schema(movie_summary_flag='positive'/'negative')
#  pydantic_json_lambda   → extract → plain string 'positive' or 'negative'
#  conditional_chain      → route to LinkedIn or Instagram chain based on the string
final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain


In [ ]:
# Test it — 'I loved this KGF movie' should classify as positive → LinkedIn post
final_orchestrator.invoke({"input": "I loved this KGF movie"})


In [ ]:
# Try a negative review → should route to Instagram branch
final_orchestrator.invoke({"input": "The movie was terrible and boring"})
